In [1]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
print('Not connected to a GPU' if gpu_info.find('failed') >= 0 else gpu_info)

Sun Aug 30 17:40:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import psutil
ram_gb = psutil.virtual_memory().total / 1e9
print(f'RAM disponible: {ram_gb:.1f} GB')
print('Not using a high-RAM runtime' if ram_gb < 20 else 'RAM alta activa')

RAM disponible: 54.8 GB
RAM alta activa


### 1. Descarga e importaciones

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, torch, gc
!pip install pytorch-forecasting pytorch-lightning lightning -q

import torch

_torch_load_original = torch.serialization.load  # referencia estable — nunca se sobrescribe, a diferencia de torch.load
def _torch_load_patched(*args, **kwargs):
    kwargs['weights_only'] = False
    return _torch_load_original(*args, **kwargs)
torch.load = _torch_load_patched

import pytorch_forecasting.data.encoders as pf_encoders

torch.serialization.add_safe_globals([
    pf_encoders.GroupNormalizer,
    pf_encoders.TorchNormalizer,
    pf_encoders.NaNLabelEncoder,
    pf_encoders.EncoderNormalizer,
    pf_encoders.MultiNormalizer,
])

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 68.3 MB/s eta 0:00:00


### 2. Carga y preparación

In [4]:
esqueleto = pd.read_parquet('/content/drive/MyDrive/TFM/favorita_panel_corregido.parquet')
if 'type' in esqueleto.columns:
    esqueleto = esqueleto.rename(columns={'type': 'store_type'})
for col in ['store_nbr', 'item_nbr', 'city', 'state', 'store_type', 'cluster', 'family', 'class', 'perishable']:
    esqueleto[col] = esqueleto[col].astype(str)

volumen_por_serie = esqueleto.groupby(['store_nbr', 'item_nbr'])['unit_sales'].transform('mean')
esqueleto['peso_muestra'] = 1 / np.sqrt(volumen_por_serie + 0.1)
esqueleto['peso_muestra'] = esqueleto['peso_muestra'] / esqueleto['peso_muestra'].mean()

fecha_max = esqueleto['date'].max()
test_inicio = fecha_max - pd.Timedelta(days=60)
val_inicio = test_inicio - pd.Timedelta(days=60)
train = esqueleto[esqueleto['date'] < val_inicio]
training_cutoff = train['time_idx'].max()

CUANTILES = [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]
max_encoder_length = 90
max_prediction_length = 30

training = TimeSeriesDataSet(
    esqueleto[esqueleto.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="unit_sales",
    group_ids=["store_nbr", "item_nbr"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=["city", "state", "store_type", "cluster", "family", "class", "perishable"],
    time_varying_known_reals=["time_idx", "year", "month", "day_of_week", "is_weekend",
                               "es_feriado", "onpromotion", "edad"],
    time_varying_unknown_reals=["unit_sales", "dcoilwtico"],
    target_normalizer=GroupNormalizer(groups=["store_nbr", "item_nbr"], transformation="log1p"),
    weight="peso_muestra",
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=False,
)

validation = TimeSeriesDataSet.from_dataset(
    training, esqueleto, min_prediction_idx=training_cutoff + 1, stop_randomization=True
)

batch_size = 128
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=0)

/usr/local/lib/python3.13/dist-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1861: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 88 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__store_nbr': '1', '__group_id__item_nbr': '1428779'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2002136'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2027777'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2027827'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2053590'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2053610'}, {'__group_id__store_nbr': '1', '__group_id__item_nbr': '2053614'}, {'__group_id__store_nbr': '10', '__group_id__item_nbr': '2033805'}, {'__group_id__store_nbr': '10', '__group_id__item_nbr': '2053590'}, {'__group_id__store_nbr': '1

In [5]:
# Verificación 1: confirmar copia completa
assert esqueleto.shape[0] == 4804366, f"Número de filas inesperado: {esqueleto.shape[0]} — revisa la copia del archivo"

columnas_esperadas = {'store_nbr', 'item_nbr', 'city', 'state', 'store_type', 'cluster', 'family', 'class',
                       'perishable', 'date', 'year', 'month', 'day_of_week', 'is_weekend', 'es_feriado',
                       'dcoilwtico', 'unit_sales', 'onpromotion', 'time_idx', 'edad'}
faltantes = columnas_esperadas - set(esqueleto.columns)
assert not faltantes, f"Faltan columnas necesarias: {faltantes}"

print('Panel verificado:', esqueleto.shape)

Panel verificado: (4804366, 23)


### 3. Configuración del modelo

In [6]:
#configuración t3 (hidden_size=30, batches=400)
tft_t3 = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=30,
    attention_head_size=1,
    dropout=0.1,
    hidden_continuous_size=15,
    loss=QuantileLoss(quantiles=CUANTILES),
    log_interval=10,
    reduce_on_plateau_patience=4,
)
print(f'Parámetros de t3: {tft_t3.size() / 1e3:.1f}k')

early_stop = EarlyStopping(monitor="val_loss", patience=5, mode="min")
lr_monitor = LearningRateMonitor()
logger = CSVLogger('/content/drive/MyDrive/TFM/lightning_logs', name="tft_t3_v2")

checkpoint_callback = ModelCheckpoint(
    dirpath='/content/drive/MyDrive/TFM/checkpoints_tft_t3_v2',
    filename='{epoch:02d}-{val_loss:.3f}',
    save_top_k=-1,
    every_n_epochs=1,
)

trainer_t3 = pl.Trainer(
    max_epochs=30, accelerator="auto", enable_model_summary=True,
    gradient_clip_val=0.1, limit_train_batches=400,
    limit_val_batches=300,   #acota la validación
    callbacks=[early_stop, lr_monitor, checkpoint_callback], logger=logger,
)

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to th

Parámetros de t3: 91.5k


In [7]:
# Verificación 2
assert torch.cuda.is_available(), "No hay GPU asignada en este entorno — cambia el tipo de entorno de ejecución antes de continuar"

### 4. Entrenamiento TFT - t3

In [8]:
import glob, re, os

ckpt_dir = '/content/drive/MyDrive/TFM/checkpoints_tft_t3_v2'
ckpts = glob.glob(os.path.join(ckpt_dir, '*.ckpt'))

def extraer_epoca(path):
    m = re.search(r'epoch=(\d+)', os.path.basename(path))
    return int(m.group(1)) if m else -1

if ckpts:
    reanudar = max(ckpts, key=extraer_epoca)
    print(f'Retomando desde: {reanudar} (época {extraer_epoca(reanudar)})')
else:
    reanudar = None
    print('No se encontró checkpoint previo — empieza desde la época 0')

print('GPU disponible:', torch.cuda.is_available())

trainer_t3.fit(tft_t3, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader, ckpt_path=reanudar)

trainer_t3.save_checkpoint('/content/drive/MyDrive/TFM/tft_t3_v2.ckpt')
print('Checkpoint final guardado: tft_t3_v2.ckpt')

del tft_t3, trainer_t3
gc.collect()
torch.cuda.empty_cache()

No se encontró checkpoint previo — empieza desde la época 0
GPU disponible: True


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │    275 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    420 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  6.5 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 20.4 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 16.4 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  3.8 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  3.8 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  3.8 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  3.8 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  7.4 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  7.4 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  1.9 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     60 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  4.7 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  3.7 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  1.9 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  3.8 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  1.9 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    217 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 91.5 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 91.5 K                                                                                               
Total estimated model params size (MB): 0.366                                                                      
Modules in train mode: 526                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

INFO: `weights_only` was not set, defaulting to `False`.
INFO:lightning.pytorch.trainer.connectors.checkpoint_connector:`weights_only` was not set, defaulting to `False`.


Checkpoint final guardado: tft_t3_v2.ckpt


###5. Revisión

In [9]:
import pandas as pd, glob

def revisar_curva(nombre_modelo, carpeta='/content/drive/MyDrive/TFM/lightning_logs'):
    archivos = sorted(glob.glob(f'{carpeta}/{nombre_modelo}/version_*/metrics.csv'))
    print(f'{nombre_modelo}: {len(archivos)} fragmento(s) encontrados')
    if not archivos:
        print('  No se encontraron logs para este modelo.')
        return None

    partes = []
    for i, archivo in enumerate(archivos):
        df = pd.read_csv(archivo)
        df['fragmento'] = i
        partes.append(df)

    log = pd.concat(partes, ignore_index=True)
    val_curve = log[['epoch', 'val_loss', 'fragmento']].dropna(subset=['val_loss'])
    val_curve = val_curve.sort_values(['epoch', 'fragmento']).drop_duplicates(subset='epoch', keep='last')
    print(val_curve.to_string(index=False))
    return val_curve

curva_t3_v2 = revisar_curva('tft_t3_v2')

tft_t3_v2: 1 fragmento(s) encontrados
 epoch  val_loss  fragmento
   0.0  0.937442          0
   1.0  0.949538          0
   2.0  0.936917          0
   3.0  0.937050          0
   4.0  0.937349          0
   5.0  0.934786          0
   6.0  0.942071          0
   7.0  0.951284          0
   8.0  0.941793          0
   9.0  0.948681          0
  10.0  0.943667          0
